# Day 13-14: 综合实战 —— 从基座模型到 Agent 的完整训练 Pipeline

**目标：** 将前 12 天所学串联，完成 **Base → SFT → DPO → RL(GRPO)** 的完整 Agent 模型训练流程。

**最终产出：** 一个能够使用工具（Function Calling）的 Agent 模型。

---
## 第一部分：14天知识地图与回顾

### 完整学习路线

```
Day 1-2       Day 3-4        Day 5-6       Day 7-8       Day 9-10      Day 11-12     Day 13-14
┌─────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
│Tokenizer│→ │ 模型架构 │→ │   SFT    │→ │   DPO    │→ │   RL     │→ │ Agent RL │→ │ 综合实战 │
│& 数据   │  │Transformer│  │有监督微调│  │偏好对齐  │  │PPO/GRPO  │  │工具调用  │  │完整Pipeline│
└─────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘
     │              │            │             │             │             │              │
  BPE/分词      Attention     LoRA微调     偏好学习      强化学习      环境交互      端到端训练
  数据格式      前向传播     Loss Masking  无需RM       组相对优势    奖励设计     Base→Agent
```

### 各阶段核心知识点

| 阶段 | 天数 | 核心内容 | 关键技术 |
|------|------|----------|----------|
| 数据基础 | Day 1-2 | Tokenizer、数据预处理 | BPE、SentencePiece、ChatML 格式 |
| 模型架构 | Day 3-4 | Transformer、注意力机制 | Multi-Head Attention、RoPE、KV Cache |
| SFT | Day 5-6 | 有监督微调 | LoRA、Loss Masking、学习率调度 |
| DPO | Day 7-8 | 直接偏好优化 | 偏好对、β参数、参考模型 |
| RL | Day 9-10 | 强化学习微调 | PPO、GRPO、奖励模型 |
| Agent RL | Day 11-12 | Agent 强化学习 | 工具调用、环境交互、轨迹奖励 |
| **综合实战** | **Day 13-14** | **完整 Pipeline** | **Base → SFT → DPO → GRPO** |

### 今天的目标

我们将把前面所有知识串联起来，完成一个 **端到端** 的 Agent 训练 Pipeline：

1. **Stage 1 - SFT**：用高质量 Agent 对话数据微调基座模型，教会 Function Calling 格式
2. **Stage 2 - DPO**：用好/坏轨迹对比，让模型偏好正确的工具使用方式
3. **Stage 3 - GRPO（可选）**：在模拟环境中用强化学习继续优化
4. **综合评估**：对比各阶段模型的能力变化

```
Qwen2-0.5B ──SFT──→ 学会格式 ──DPO──→ 学会偏好 ──GRPO──→ 环境优化 ──→ Agent 模型
 (基座)        │                │                │                │
           对话数据          偏好对           环境奖励        测试评估
```

### 技术栈准备

| 组件 | 用途 |
|------|------|
| `transformers` | 模型加载与推理 |
| `torch` | 训练框架 |
| `peft` | LoRA 高效微调 |
| `matplotlib` | 评估可视化 |
| Qwen2-0.5B | 基座模型（小巧高效） |

In [ ]:
import json
import os
import sys
import copy
import time
import random
import math
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# 确保可以导入项目模块
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 第二部分：Agent 训练数据准备

### Function Calling 格式详解

Function Calling 是 Agent 的核心能力 —— 模型通过 **结构化的 JSON** 调用外部工具，而非生成自然语言猜测。

#### 工具定义格式（JSON Schema）

每个工具用标准 JSON Schema 描述其名称、功能和参数：

```json
{
  "type": "function",
  "function": {
    "name": "web_search",
    "description": "在互联网上搜索信息",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {"type": "string", "description": "搜索关键词"},
        "max_results": {"type": "integer", "description": "最大返回结果数", "default": 5}
      },
      "required": ["query"]
    }
  }
}
```

#### 工具调用格式

模型需要生成包含 `tool_calls` 的 assistant 消息：

```json
{
  "role": "assistant",
  "content": null,
  "tool_calls": [{
    "id": "call_001",
    "type": "function",
    "function": {
      "name": "web_search",
      "arguments": "{\"query\": \"大语言模型最新进展\"}"
    }
  }]
}
```

环境返回工具结果后，模型再根据结果生成最终回答。这就是 Agent 的 **"思考→调用→观察→回答"** 循环。

### 定义 5 种工具

我们为 Agent 定义 5 种常用工具：
- **web_search** - 互联网搜索
- **calculator** - 数学计算
- **file_read** - 读取文件
- **file_write** - 写入文件
- **get_weather** - 获取天气

In [ ]:
# 工具定义 —— Agent 可以使用的外部工具
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "在互联网上搜索信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "搜索关键词"},
                    "max_results": {"type": "integer", "description": "最大返回结果数", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "执行数学计算",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "数学表达式，如 '2+3*4'"},
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "file_read",
            "description": "读取文件内容",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "文件路径"},
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "file_write",
            "description": "将内容写入文件",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "文件路径"},
                    "content": {"type": "string", "description": "要写入的内容"},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"},
                },
                "required": ["city"],
            },
        },
    },
]

print(f"共定义 {len(TOOL_DEFINITIONS)} 种工具：")
for tool in TOOL_DEFINITIONS:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  - {fn['name']}: {fn['description']}  参数: {params}")

In [ ]:
# 模拟工具返回结果（用于生成训练数据）
MOCK_SEARCH_RESULTS = {
    "大语言模型": [
        {"title": "GPT-5 发布，推理能力大幅提升", "snippet": "OpenAI发布GPT-5，在数学推理和代码生成上表现优异..."},
        {"title": "开源模型 Qwen2.5 性能逼近闭源", "snippet": "阿里发布Qwen2.5系列，在多项基准上接近GPT-4..."},
    ],
    "量子计算": [
        {"title": "Google 量子纠错取得突破", "snippet": "Willow芯片实现低于阈值的量子纠错..."},
    ],
}

MOCK_WEATHER = {
    "北京": {"temp": "12°C", "condition": "晴", "wind": "北风3级", "humidity": "35%"},
    "上海": {"temp": "18°C", "condition": "多云", "wind": "东南风2级", "humidity": "65%"},
    "深圳": {"temp": "25°C", "condition": "阴", "wind": "南风2级", "humidity": "78%"},
    "成都": {"temp": "15°C", "condition": "小雨", "wind": "微风", "humidity": "82%"},
    "杭州": {"temp": "16°C", "condition": "晴转多云", "wind": "东风2级", "humidity": "60%"},
}


def mock_tool_result(tool_name: str, arguments: Dict) -> str:
    """模拟工具调用返回结果"""
    if tool_name == "web_search":
        query = arguments.get("query", "")
        for key, results in MOCK_SEARCH_RESULTS.items():
            if key in query:
                return json.dumps(results, ensure_ascii=False)
        return json.dumps([{"title": f"关于{query}的最新研究", "snippet": f"{query}领域近期取得重要进展..."}], ensure_ascii=False)
    elif tool_name == "calculator":
        expr = arguments.get("expression", "0")
        try:
            result = eval(expr, {"__builtins__": {}}, {"sqrt": math.sqrt, "log": math.log10, "pi": math.pi})
            return str(result)
        except Exception:
            return "计算错误"
    elif tool_name == "file_read":
        return f"[文件内容] 这是 {arguments.get('path', '')} 的模拟内容。包含一些数据和分析结果。"
    elif tool_name == "file_write":
        return f"文件 {arguments.get('path', '')} 写入成功。"
    elif tool_name == "get_weather":
        city = arguments.get("city", "北京")
        weather = MOCK_WEATHER.get(city, {"temp": "20°C", "condition": "晴", "wind": "微风", "humidity": "50%"})
        return json.dumps(weather, ensure_ascii=False)
    return "未知工具"

# 测试工具
print("搜索测试:", mock_tool_result("web_search", {"query": "大语言模型"})[:80], "...")
print("计算测试:", mock_tool_result("calculator", {"expression": "(15+27)*3-18/6"}))
print("天气测试:", mock_tool_result("get_weather", {"city": "北京"}))

### 生成 SFT 训练数据

SFT 数据是 **高质量的 Agent 对话轨迹**，教会模型：
- 何时应该调用工具（而非直接回答）
- 如何正确构造工具调用的 JSON 格式
- 如何利用工具返回结果生成最终回答

一条好的 SFT 样本包含完整的 **"用户提问 → 工具调用 → 工具结果 → 最终回答"** 链条。

In [ ]:
# 任务模板
TASK_TEMPLATES = [
    {
        "id": "search_and_summarize",
        "user_query": "帮我搜索一下 {topic} 的最新进展，并总结要点。",
        "topics": ["大语言模型", "量子计算", "可控核聚变", "脑机接口", "自动驾驶"],
        "system_prompt": "你是一个有用的AI助手，可以使用工具来帮助用户完成任务。",
    },
    {
        "id": "math_problem",
        "user_query": "请计算 {expression}，并解释计算过程。",
        "expressions": ["(15 + 27) * 3 - 18 / 6", "2**10 + 3**5", "sqrt(144) + log(100)"],
        "system_prompt": "你是一个有用的AI助手，可以使用工具来帮助用户完成任务。",
    },
    {
        "id": "file_operation",
        "user_query": "读取 {filename} 的内容，然后将摘要写入 summary.txt。",
        "filenames": ["report.txt", "data.csv", "notes.md"],
        "system_prompt": "你是一个有用的AI助手，可以使用工具来帮助用户完成任务。",
    },
    {
        "id": "weather_plan",
        "user_query": "查看 {city} 的天气，帮我规划明天的出行方案。",
        "cities": ["北京", "上海", "深圳", "成都", "杭州"],
        "system_prompt": "你是一个有用的AI助手，可以使用工具来帮助用户完成任务。",
    },
    {
        "id": "multi_step_research",
        "user_query": "搜索 {topic} 相关信息，计算关键数据，并将报告保存到 report.txt。",
        "topics": ["中国GDP增长率", "全球碳排放数据", "AI行业融资趋势"],
        "system_prompt": "你是一个有用的AI助手，可以使用工具来帮助用户完成任务。",
    },
]


def generate_good_trajectory(template: Dict, filled_query: str) -> List[Dict]:
    """生成高质量 Agent 轨迹（好样本）"""
    messages = [
        {"role": "system", "content": template["system_prompt"]},
        {"role": "user", "content": filled_query},
    ]
    task_id = template["id"]

    if task_id == "search_and_summarize":
        topic = filled_query.split("搜索一下 ")[1].split(" 的")[0]
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_001", "type": "function", "function": {"name": "web_search", "arguments": json.dumps({"query": f"{topic} 最新进展 2024", "max_results": 5}, ensure_ascii=False)}}]})
        messages.append({"role": "tool", "tool_call_id": "call_001", "content": mock_tool_result("web_search", {"query": topic})})
        messages.append({"role": "assistant", "content": f"根据搜索结果，以下是{topic}的最新进展要点：\n\n1. 技术突破：该领域近期取得了重要技术突破，多项关键指标显著提升。\n2. 产业应用：越来越多的企业开始将{topic}技术落地到实际场景中。\n3. 未来趋势：专家预测{topic}将在未来几年持续快速发展。\n\n如需更详细的信息，我可以进一步搜索。"})

    elif task_id == "math_problem":
        expr = filled_query.split("请计算 ")[1].split("，")[0]
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_001", "type": "function", "function": {"name": "calculator", "arguments": json.dumps({"expression": expr})}}]})
        calc_result = mock_tool_result("calculator", {"expression": expr})
        messages.append({"role": "tool", "tool_call_id": "call_001", "content": calc_result})
        messages.append({"role": "assistant", "content": f"计算结果为 {calc_result}。\n\n计算过程：\n- 表达式：{expr}\n- 按照运算优先级，先计算括号和幂运算，再计算乘除，最后计算加减。"})

    elif task_id == "file_operation":
        filename = filled_query.split("读取 ")[1].split(" 的")[0]
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_001", "type": "function", "function": {"name": "file_read", "arguments": json.dumps({"path": filename})}}]})
        messages.append({"role": "tool", "tool_call_id": "call_001", "content": mock_tool_result("file_read", {"path": filename})})
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_002", "type": "function", "function": {"name": "file_write", "arguments": json.dumps({"path": "summary.txt", "content": f"{filename}的摘要：文件包含重要数据和分析结果，核心要点已提取。"}, ensure_ascii=False)}}]})
        messages.append({"role": "tool", "tool_call_id": "call_002", "content": mock_tool_result("file_write", {"path": "summary.txt"})})
        messages.append({"role": "assistant", "content": f"已完成操作：\n1. 读取了 {filename} 的内容\n2. 提取了关键信息并生成摘要\n3. 将摘要写入了 summary.txt"})

    elif task_id == "weather_plan":
        city = filled_query.split("查看 ")[1].split(" 的")[0]
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_001", "type": "function", "function": {"name": "get_weather", "arguments": json.dumps({"city": city}, ensure_ascii=False)}}]})
        weather_result = mock_tool_result("get_weather", {"city": city})
        messages.append({"role": "tool", "tool_call_id": "call_001", "content": weather_result})
        weather = json.loads(weather_result)
        messages.append({"role": "assistant", "content": f"{city}明天天气预报：{weather['condition']}，气温{weather['temp']}，{weather['wind']}。\n\n出行建议：\n- 穿着：根据{weather['temp']}的气温，建议适当增减衣物。\n- 出行方式：{'建议携带雨具' if '雨' in weather['condition'] else '适合户外活动'}。\n- 注意事项：湿度{weather['humidity']}，注意补水。"})

    elif task_id == "multi_step_research":
        topic = filled_query.split("搜索 ")[1].split(" 相关")[0]
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_001", "type": "function", "function": {"name": "web_search", "arguments": json.dumps({"query": topic, "max_results": 5}, ensure_ascii=False)}}]})
        messages.append({"role": "tool", "tool_call_id": "call_001", "content": mock_tool_result("web_search", {"query": topic})})
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_002", "type": "function", "function": {"name": "calculator", "arguments": json.dumps({"expression": "100 * 1.05 ** 5"})}}]})
        calc_result = mock_tool_result("calculator", {"expression": "100 * 1.05 ** 5"})
        messages.append({"role": "tool", "tool_call_id": "call_002", "content": calc_result})
        report_content = f"# {topic} 研究报告\n\n## 搜索结果摘要\n基于最新数据...\n\n## 数据分析\n预测值: {calc_result}\n"
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_003", "type": "function", "function": {"name": "file_write", "arguments": json.dumps({"path": "report.txt", "content": report_content}, ensure_ascii=False)}}]})
        messages.append({"role": "tool", "tool_call_id": "call_003", "content": mock_tool_result("file_write", {"path": "report.txt"})})
        messages.append({"role": "assistant", "content": f"已完成{topic}的研究报告：\n1. 搜索了最新相关信息\n2. 对关键数据进行了计算分析\n3. 将完整报告保存到 report.txt"})

    return messages


def generate_sft_data(num_samples: int = 50, seed: int = 42) -> List[Dict]:
    """生成 SFT 训练数据"""
    random.seed(seed)
    dataset = []
    for i in range(num_samples):
        template = random.choice(TASK_TEMPLATES)
        task_id = template["id"]
        if task_id == "search_and_summarize":
            query = template["user_query"].format(topic=random.choice(template["topics"]))
        elif task_id == "math_problem":
            query = template["user_query"].format(expression=random.choice(template["expressions"]))
        elif task_id == "file_operation":
            query = template["user_query"].format(filename=random.choice(template["filenames"]))
        elif task_id == "weather_plan":
            query = template["user_query"].format(city=random.choice(template["cities"]))
        elif task_id == "multi_step_research":
            query = template["user_query"].format(topic=random.choice(template["topics"]))
        else:
            continue
        messages = generate_good_trajectory(template, query)
        dataset.append({"id": f"sft_{i:04d}", "messages": messages, "tools": TOOL_DEFINITIONS})
    return dataset


# 生成 SFT 数据并展示样例
sft_data = generate_sft_data(num_samples=50)
print(f"生成 SFT 数据: {len(sft_data)} 条")
print(f"\n{'='*60}")
print("SFT 样例展示（前3条）：")
print(f"{'='*60}")
for sample in sft_data[:3]:
    print(f"\n--- {sample['id']} ---")
    for msg in sample["messages"]:
        role = msg["role"]
        if role == "assistant" and msg.get("tool_calls"):
            tc = msg["tool_calls"][0]["function"]
            print(f"  [{role}] → 调用工具: {tc['name']}({tc['arguments'][:60]}...)")
        elif role == "tool":
            print(f"  [{role}] → {msg['content'][:60]}...")
        else:
            content = msg.get("content", "")
            if content:
                print(f"  [{role}] {content[:80]}...")

### 生成 DPO 偏好数据

DPO 需要 **偏好对**：同一个 prompt，一条好轨迹（chosen）和一条坏轨迹（rejected）。

**4 种典型的坏轨迹模式：**

| 模式 | 说明 | 示例 |
|------|------|------|
| **wrong_tool** | 选错工具 | 该搜索时用了计算器 |
| **bad_params** | 参数错误 | 搜索 query 为空 |
| **no_tool** | 不用工具 | 该调工具时直接编造回答 |
| **hallucinate** | 忽视结果 | 调了工具但无视返回，编造数据 |

In [ ]:
def generate_bad_trajectory(template: Dict, filled_query: str) -> List[Dict]:
    """生成低质量 Agent 轨迹（坏样本）"""
    messages = [
        {"role": "system", "content": template["system_prompt"]},
        {"role": "user", "content": filled_query},
    ]
    # 随机选一种坏模式
    bad_pattern = random.choice(["wrong_tool", "bad_params", "no_tool", "hallucinate"])

    if bad_pattern == "wrong_tool":
        # 该搜索时却用计算器
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_bad_001", "type": "function", "function": {"name": "calculator", "arguments": json.dumps({"expression": "1+1"})}}]})
        messages.append({"role": "tool", "tool_call_id": "call_bad_001", "content": "2"})
        messages.append({"role": "assistant", "content": "结果是2。我已经帮你处理了。"})

    elif bad_pattern == "bad_params":
        # 参数为空
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_bad_001", "type": "function", "function": {"name": "web_search", "arguments": json.dumps({"query": ""})}}]})
        messages.append({"role": "tool", "tool_call_id": "call_bad_001", "content": "搜索查询不能为空"})
        messages.append({"role": "assistant", "content": "搜索完成了，这里是结果。"})

    elif bad_pattern == "no_tool":
        # 不用工具直接编造
        messages.append({"role": "assistant", "content": "我来直接回答你的问题。根据我的知识，这个问题的答案是...... （这里是一段看似合理但可能过时或错误的回答，因为没有使用工具获取最新信息）"})

    elif bad_pattern == "hallucinate":
        # 调了工具但编造回答
        tool_name = random.choice(["web_search", "calculator", "get_weather"])
        args = {"query": "随便搜搜"} if tool_name == "web_search" else ({"expression": "1+1"} if tool_name == "calculator" else {"city": "北京"})
        messages.append({"role": "assistant", "content": None, "tool_calls": [{"id": "call_bad_001", "type": "function", "function": {"name": tool_name, "arguments": json.dumps(args, ensure_ascii=False)}}]})
        messages.append({"role": "tool", "tool_call_id": "call_bad_001", "content": mock_tool_result(tool_name, args)})
        messages.append({"role": "assistant", "content": "根据最新数据显示，全球GDP增长了500%，AI已经完全取代了所有人类工作。（注意：这是完全编造的信息，与工具返回的结果无关）"})

    return messages


def generate_dpo_data(num_samples: int = 30, seed: int = 42) -> List[Dict]:
    """生成 DPO 偏好对数据"""
    random.seed(seed)
    dataset = []
    for i in range(num_samples):
        template = random.choice(TASK_TEMPLATES)
        task_id = template["id"]
        if task_id == "search_and_summarize":
            query = template["user_query"].format(topic=random.choice(template["topics"]))
        elif task_id == "math_problem":
            query = template["user_query"].format(expression=random.choice(template["expressions"]))
        elif task_id == "file_operation":
            query = template["user_query"].format(filename=random.choice(template["filenames"]))
        elif task_id == "weather_plan":
            query = template["user_query"].format(city=random.choice(template["cities"]))
        elif task_id == "multi_step_research":
            query = template["user_query"].format(topic=random.choice(template["topics"]))
        else:
            continue
        good_traj = generate_good_trajectory(template, query)
        bad_traj = generate_bad_trajectory(template, query)
        prompt_messages = [m for m in good_traj if m["role"] in ("system", "user")]
        chosen_messages = [m for m in good_traj if m["role"] not in ("system", "user")]
        rejected_messages = [m for m in bad_traj if m["role"] not in ("system", "user")]
        dataset.append({"id": f"dpo_{i:04d}", "prompt": prompt_messages, "chosen": chosen_messages, "rejected": rejected_messages, "tools": TOOL_DEFINITIONS})
    return dataset


# 生成 DPO 数据并展示
dpo_data = generate_dpo_data(num_samples=30)
print(f"生成 DPO 数据: {len(dpo_data)} 条")

# 展示一组 chosen vs rejected 对比
sample = dpo_data[0]
print(f"\n{'='*60}")
print(f"DPO 样例对比 ({sample['id']})")
print(f"{'='*60}")
print(f"\n[Prompt]")
for m in sample["prompt"]:
    print(f"  {m['role']}: {m.get('content', '')[:60]}")
print(f"\n[Chosen - 好轨迹] ({len(sample['chosen'])} 轮)")
for m in sample["chosen"]:
    if m.get("tool_calls"):
        print(f"  assistant → 工具调用: {m['tool_calls'][0]['function']['name']}")
    elif m["role"] == "tool":
        print(f"  tool → {m['content'][:50]}...")
    else:
        print(f"  {m['role']}: {m.get('content', '')[:60]}...")
print(f"\n[Rejected - 坏轨迹] ({len(sample['rejected'])} 轮)")
for m in sample["rejected"]:
    if m.get("tool_calls"):
        print(f"  assistant → 工具调用: {m['tool_calls'][0]['function']['name']}")
    elif m["role"] == "tool":
        print(f"  tool → {m['content'][:50]}...")
    else:
        print(f"  {m['role']}: {m.get('content', '')[:60]}...")

---
## 第三部分：Stage 1 - SFT 有监督微调

### SFT 要点回顾

**SFT（Supervised Fine-Tuning）** 是训练 Agent 的第一步，核心目标是让基座模型学会：
1. **对话格式** —— system / user / assistant / tool 的多轮结构
2. **Function Calling** —— 正确生成 `tool_calls` JSON
3. **工具结果利用** —— 根据 tool 返回的内容生成有用的回答

#### 为什么需要 Loss Masking？

在 SFT 中，我们 **只想让模型学会 "如何回复"**，而不是学会 "如何提问"。因此：
- **assistant 回复部分**：正常计算 loss（模型需要学习的部分）
- **system / user / tool 部分**：label 设为 -100，被 CrossEntropy 忽略

这样模型只会从 assistant 的行为中学习，不会把 user 的提问方式也学进去。

### LoRA 高效微调

全量微调 Qwen2-0.5B 的所有参数需要较多显存。**LoRA（Low-Rank Adaptation）** 只训练少量的低秩矩阵：

```
原始权重 W (d × d)  →  W + ΔW = W + A·B  (A: d×r, B: r×d, r << d)
```

| 参数 | 说明 | 推荐值 |
|------|------|--------|
| `r` | 秩（低秩矩阵的维度） | 8-64 |
| `alpha` | 缩放因子 | 通常 = 2r |
| `target_modules` | 要应用 LoRA 的层 | q_proj, v_proj 等 |

In [ ]:
# 训练配置
@dataclass
class TrainingConfig:
    """训练超参数配置"""
    model_name: str = "Qwen/Qwen2-0.5B"
    # SFT 阶段
    sft_epochs: int = 3
    sft_lr: float = 2e-5
    sft_batch_size: int = 2
    sft_max_length: int = 512
    # DPO 阶段
    dpo_epochs: int = 2
    dpo_lr: float = 5e-6
    dpo_batch_size: int = 2
    dpo_beta: float = 0.1  # DPO 温度系数
    dpo_max_length: int = 512
    # GRPO 阶段
    grpo_epochs: int = 1
    grpo_lr: float = 1e-6
    grpo_batch_size: int = 4
    grpo_group_size: int = 4  # 每个 prompt 采样的回复数
    grpo_clip_eps: float = 0.2
    grpo_kl_coeff: float = 0.05
    # 通用
    output_dir: str = "checkpoints"
    seed: int = 42
    gradient_accumulation_steps: int = 4
    max_grad_norm: float = 1.0
    use_fp16: bool = True
    device: str = "auto"

    def get_device(self) -> torch.device:
        if self.device == "auto":
            return torch.device("cuda" if torch.cuda.is_available() else "cpu")
        return torch.device(self.device)

config = TrainingConfig()
device = config.get_device()
print(f"训练配置：")
print(f"  模型: {config.model_name}")
print(f"  设备: {device}")
print(f"  SFT: {config.sft_epochs} epochs, lr={config.sft_lr}")
print(f"  DPO: {config.dpo_epochs} epochs, lr={config.dpo_lr}, β={config.dpo_beta}")
print(f"  GRPO: {config.grpo_epochs} epochs, group_size={config.grpo_group_size}")

In [ ]:
# 加载基座模型
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"加载基座模型: {config.model_name}")
tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.float16 if config.use_fp16 and device.type == "cuda" else torch.float32,
    trust_remote_code=True,
)

num_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {num_params / 1e6:.1f}M")
print(f"词表大小: {tokenizer.vocab_size}")
print(f"模型架构: {model.config.model_type}")

In [ ]:
# SFT 数据集类
class AgentSFTDataset(Dataset):
    """
    SFT 数据集：将多轮对话转为可训练的 token 序列。
    关键：只对 assistant 回复部分计算 loss（padding 部分 label = -100）
    """
    def __init__(self, data: List[Dict], tokenizer, max_length: int = 512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        messages = item["messages"]
        # 拼接多轮对话为文本
        text_parts = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "")
            if role == "assistant" and msg.get("tool_calls"):
                tc = msg["tool_calls"][0]["function"]
                content = f"<tool_call>{json.dumps({'name': tc['name'], 'arguments': json.loads(tc['arguments'])}, ensure_ascii=False)}</tool_call>"
            if content:
                text_parts.append(f"<|{role}|>\n{content}")
        text = "\n".join(text_parts) + self.tokenizer.eos_token

        encoding = self.tokenizer(
            text, max_length=self.max_length, truncation=True,
            padding="max_length", return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100  # padding 部分不计算 loss
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

# 创建 SFT 数据集
sft_dataset = AgentSFTDataset(sft_data, tokenizer, config.sft_max_length)
print(f"SFT 数据集大小: {len(sft_dataset)}")

# 检查一条样本
sample = sft_dataset[0]
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"有效 token 数: {(sample['attention_mask'] == 1).sum().item()}")
print(f"需要计算 loss 的 token 数: {(sample['labels'] != -100).sum().item()}")

In [ ]:
# SFT 训练
def train_sft(model, tokenizer, config, sft_data):
    """Stage 1: 有监督微调"""
    print("\n" + "=" * 60)
    print("Stage 1: SFT（有监督微调）")
    print("=" * 60)

    device = config.get_device()
    model = model.to(device)
    model.train()

    dataset = AgentSFTDataset(sft_data, tokenizer, config.sft_max_length)
    dataloader = DataLoader(dataset, batch_size=config.sft_batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.sft_lr, weight_decay=0.01)

    # 学习率调度：线性 warmup + cosine decay
    total_steps = len(dataloader) * config.sft_epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=config.sft_lr, total_steps=total_steps, pct_start=0.1
    )

    sft_losses = []  # 记录训练损失
    global_step = 0

    for epoch in range(config.sft_epochs):
        epoch_loss = 0.0
        num_batches = 0
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / config.gradient_accumulation_steps
            loss.backward()

            if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            current_loss = outputs.loss.item()
            epoch_loss += current_loss
            num_batches += 1
            sft_losses.append(current_loss)

            if batch_idx % 10 == 0:
                print(f"  Epoch {epoch+1}/{config.sft_epochs}, "
                      f"Batch {batch_idx}/{len(dataloader)}, "
                      f"Loss: {current_loss:.4f}")

        avg_loss = epoch_loss / max(num_batches, 1)
        print(f"  Epoch {epoch+1} 平均 Loss: {avg_loss:.4f}")

    # 保存 SFT 检查点
    sft_dir = os.path.join(config.output_dir, "sft")
    os.makedirs(sft_dir, exist_ok=True)
    model.save_pretrained(sft_dir)
    tokenizer.save_pretrained(sft_dir)
    print(f"  SFT 模型已保存到 {sft_dir}")

    return model, sft_losses

# 执行 SFT 训练
model, sft_losses = train_sft(model, tokenizer, config, sft_data)

In [ ]:
# 测试 SFT 后模型是否学会了 function calling 格式
model.eval()
test_prompts = [
    "帮我搜索一下大语言模型的最新进展，并总结要点。",
    "请计算 (15 + 27) * 3 - 18 / 6，并解释计算过程。",
    "查看北京的天气，帮我规划明天的出行方案。",
]

print("SFT 后模型测试：")
print("=" * 60)
for prompt in test_prompts:
    input_text = f"<|system|>\n你是一个有用的AI助手，可以使用工具来帮助用户完成任务。\n<|user|>\n{prompt}\n<|assistant|>\n"
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    has_tool_call = "<tool_call>" in response or "tool_call" in response
    print(f"\n[Query] {prompt}")
    print(f"[Response] {response[:200]}")
    print(f"[包含工具调用格式] {'是' if has_tool_call else '否'}")
    print("-" * 40)

In [ ]:
# 可视化 SFT 训练损失
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sft_losses, color="#4ECDC4", alpha=0.6, linewidth=0.8, label="每步 Loss")
# 滑动平均
window = min(10, len(sft_losses))
if len(sft_losses) >= window:
    smoothed = np.convolve(sft_losses, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(sft_losses)), smoothed, color="#FF6B6B", linewidth=2, label=f"滑动平均(window={window})")
ax.set_xlabel("训练步数")
ax.set_ylabel("Loss")
ax.set_title("Stage 1: SFT 训练损失曲线")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("sft_loss.png", dpi=150)
plt.show()
print("SFT 损失曲线已保存")

---
## 第四部分：Stage 2 - DPO 直接偏好优化

### DPO 要点回顾

**DPO（Direct Preference Optimization）** 的核心思想：
- **不需要训练奖励模型**（与 RLHF 的关键区别）
- 直接从偏好数据（chosen vs rejected）中学习
- 使模型偏好 chosen 轨迹，同时不偏离参考模型太远

#### DPO 损失函数

$$\mathcal{L}_{DPO} = -\log \sigma\left(\beta \cdot \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)$$

- $\pi_\theta$ — 正在训练的模型
- $\pi_{ref}$ — 参考模型（冻结的 SFT 模型）
- $y_w$ — chosen（好回复），$y_l$ — rejected（差回复）
- $\beta$ — 温度系数，控制偏离参考模型的程度

**直觉理解：** 让好回复相对于参考模型的「概率增量」大于坏回复的「概率增量」。

### β 参数的作用

| β 值 | 效果 | 风险 |
|------|------|------|
| β 太小 (0.01) | 模型变化大，学习快 | 可能丢失基础能力 |
| β 适中 (0.1) | 平衡学习与稳定性 | 推荐值 |
| β 太大 (1.0) | 模型几乎不变 | 学不到偏好信息 |

β 本质上是 **KL 散度约束的拉格朗日乘子**，控制策略与参考模型之间的距离。

In [ ]:
# DPO 数据集类
class AgentDPODataset(Dataset):
    """DPO 数据集：每条数据包含 chosen 和 rejected 回复"""
    def __init__(self, data: List[Dict], tokenizer, max_length: int = 512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def _messages_to_text(self, messages: List[Dict]) -> str:
        parts = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "")
            if role == "assistant" and msg.get("tool_calls"):
                tc = msg["tool_calls"][0]["function"]
                content = f"<tool_call>{json.dumps({'name': tc['name'], 'arguments': json.loads(tc['arguments'])}, ensure_ascii=False)}</tool_call>"
            if content:
                parts.append(f"<|{role}|>\n{content}")
        return "\n".join(parts)

    def __getitem__(self, idx):
        item = self.data[idx]
        prompt_text = self._messages_to_text(item["prompt"])
        chosen_text = prompt_text + "\n" + self._messages_to_text(item["chosen"]) + self.tokenizer.eos_token
        rejected_text = prompt_text + "\n" + self._messages_to_text(item["rejected"]) + self.tokenizer.eos_token

        chosen_enc = self.tokenizer(chosen_text, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")
        rejected_enc = self.tokenizer(rejected_text, max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")

        return {
            "chosen_input_ids": chosen_enc["input_ids"].squeeze(0),
            "chosen_attention_mask": chosen_enc["attention_mask"].squeeze(0),
            "rejected_input_ids": rejected_enc["input_ids"].squeeze(0),
            "rejected_attention_mask": rejected_enc["attention_mask"].squeeze(0),
        }

dpo_dataset = AgentDPODataset(dpo_data, tokenizer, config.dpo_max_length)
print(f"DPO 数据集大小: {len(dpo_dataset)}")

In [ ]:
# DPO 损失计算
def compute_dpo_loss(
    model, ref_model,
    chosen_input_ids, chosen_attention_mask,
    rejected_input_ids, rejected_attention_mask,
    beta: float = 0.1,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    计算 DPO 损失。
    L = -log σ(β * (log π_θ(y_w|x)/π_ref(y_w|x) - log π_θ(y_l|x)/π_ref(y_l|x)))
    """
    def get_log_probs(m, input_ids, attention_mask):
        with torch.no_grad() if not m.training else torch.enable_grad():
            outputs = m(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits[:, :-1, :]  # 去掉最后一个位置
            labels = input_ids[:, 1:]            # 目标是下一个 token
            log_probs = F.log_softmax(logits, dim=-1)
            token_log_probs = torch.gather(log_probs, 2, labels.unsqueeze(-1)).squeeze(-1)
            mask = attention_mask[:, 1:]
            return (token_log_probs * mask).sum(dim=-1)  # 序列级 log prob

    # 当前策略的 log prob
    policy_chosen_logps = get_log_probs(model, chosen_input_ids, chosen_attention_mask)
    policy_rejected_logps = get_log_probs(model, rejected_input_ids, rejected_attention_mask)

    # 参考策略的 log prob（不计算梯度）
    with torch.no_grad():
        ref_chosen_logps = get_log_probs(ref_model, chosen_input_ids, chosen_attention_mask)
        ref_rejected_logps = get_log_probs(ref_model, rejected_input_ids, rejected_attention_mask)

    # 计算隐式奖励
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)

    # DPO loss
    loss = -F.logsigmoid(chosen_rewards - rejected_rewards).mean()

    metrics = {
        "chosen_reward": chosen_rewards.mean().item(),
        "rejected_reward": rejected_rewards.mean().item(),
        "reward_margin": (chosen_rewards - rejected_rewards).mean().item(),
        "accuracy": (chosen_rewards > rejected_rewards).float().mean().item(),
    }
    return loss, metrics

print("DPO 损失函数已定义")

In [ ]:
# 准备参考模型（冻结的 SFT 模型副本）
print("准备 DPO 参考模型（冻结 SFT 模型副本）...")
ref_model = copy.deepcopy(model)
for param in ref_model.parameters():
    param.requires_grad = False
ref_model.eval()
print(f"参考模型已冻结，参数量: {sum(p.numel() for p in ref_model.parameters()) / 1e6:.1f}M")

In [ ]:
# DPO 训练
def train_dpo(model, ref_model, tokenizer, config, dpo_data):
    """Stage 2: DPO 直接偏好优化"""
    print("\n" + "=" * 60)
    print("Stage 2: DPO（直接偏好优化）")
    print("=" * 60)

    device = config.get_device()
    model = model.to(device)
    ref_model = ref_model.to(device)
    model.train()

    dataset = AgentDPODataset(dpo_data, tokenizer, config.dpo_max_length)
    dataloader = DataLoader(dataset, batch_size=config.dpo_batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.dpo_lr, weight_decay=0.01)

    dpo_losses = []
    dpo_accuracies = []
    dpo_margins = []

    for epoch in range(config.dpo_epochs):
        epoch_loss = 0.0
        epoch_acc = 0.0
        num_batches = 0

        for batch_idx, batch in enumerate(dataloader):
            chosen_input_ids = batch["chosen_input_ids"].to(device)
            chosen_attention_mask = batch["chosen_attention_mask"].to(device)
            rejected_input_ids = batch["rejected_input_ids"].to(device)
            rejected_attention_mask = batch["rejected_attention_mask"].to(device)

            loss, metrics = compute_dpo_loss(
                model, ref_model,
                chosen_input_ids, chosen_attention_mask,
                rejected_input_ids, rejected_attention_mask,
                beta=config.dpo_beta,
            )

            loss.backward()
            if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                optimizer.step()
                optimizer.zero_grad()

            epoch_loss += loss.item()
            epoch_acc += metrics["accuracy"]
            num_batches += 1
            dpo_losses.append(loss.item())
            dpo_accuracies.append(metrics["accuracy"])
            dpo_margins.append(metrics["reward_margin"])

            if batch_idx % 5 == 0:
                print(f"  Epoch {epoch+1}/{config.dpo_epochs}, "
                      f"Batch {batch_idx}/{len(dataloader)}, "
                      f"Loss: {loss.item():.4f}, "
                      f"Acc: {metrics['accuracy']:.2%}, "
                      f"Margin: {metrics['reward_margin']:.4f}")

        avg_loss = epoch_loss / max(num_batches, 1)
        avg_acc = epoch_acc / max(num_batches, 1)
        print(f"  Epoch {epoch+1} 平均 Loss: {avg_loss:.4f}, Accuracy: {avg_acc:.2%}")

    # 保存 DPO 检查点
    dpo_dir = os.path.join(config.output_dir, "dpo")
    os.makedirs(dpo_dir, exist_ok=True)
    model.save_pretrained(dpo_dir)
    tokenizer.save_pretrained(dpo_dir)
    print(f"  DPO 模型已保存到 {dpo_dir}")

    return model, {"losses": dpo_losses, "accuracies": dpo_accuracies, "margins": dpo_margins}

# 执行 DPO 训练
model, dpo_metrics = train_dpo(model, ref_model, tokenizer, config, dpo_data)

In [ ]:
# 对比 SFT vs SFT+DPO
model.eval()
print("SFT+DPO 后模型测试 vs SFT：")
print("=" * 60)

comparison_prompts = [
    "帮我搜索一下大语言模型的最新进展，并总结要点。",
    "请计算 2**10 + 3**5，并解释计算过程。",
]

for prompt in comparison_prompts:
    input_text = f"<|system|>\n你是一个有用的AI助手，可以使用工具来帮助用户完成任务。\n<|user|>\n{prompt}\n<|assistant|>\n"
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids, max_new_tokens=200, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n[Query] {prompt}")
    print(f"[DPO Response] {response[:250]}")
    print("-" * 40)

print("\n期望观察到的改进：")
print("  - 工具选择更准确（不会选错工具）")
print("  - 回答更完整有条理（不会敷衍）")
print("  - 减少冗余步骤（不会重复调用）")

In [ ]:
# 可视化 DPO 训练指标
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(dpo_metrics["losses"], color="#FF6B6B", alpha=0.7)
axes[0].set_title("DPO Loss")
axes[0].set_xlabel("步数")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

axes[1].plot(dpo_metrics["accuracies"], color="#4ECDC4", alpha=0.7)
axes[1].set_title("DPO Accuracy（chosen > rejected）")
axes[1].set_xlabel("步数")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.1)
axes[1].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="随机基线")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(dpo_metrics["margins"], color="#45B7D1", alpha=0.7)
axes[2].set_title("Reward Margin（chosen - rejected）")
axes[2].set_xlabel("步数")
axes[2].set_ylabel("Margin")
axes[2].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[2].grid(alpha=0.3)

plt.suptitle("Stage 2: DPO 训练指标", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("dpo_metrics.png", dpi=150)
plt.show()
print("DPO 指标图已保存")

In [ ]:
# 释放参考模型显存
del ref_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"已释放参考模型，当前显存占用: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("已释放参考模型内存")

---
## 第五部分：Stage 3 - GRPO 强化学习（可选）

### GRPO 要点回顾

**GRPO（Group Relative Policy Optimization）** 是 DeepSeek 提出的强化学习方法，核心改进：
- **无需 Critic 网络**：通过组内相对排名计算优势值，省掉了 PPO 中的 value network
- 对每个 prompt 采样 G 个回复，用环境奖励打分，标准化后得到相对优势

$$A_i = \frac{r_i - \text{mean}(r)}{\text{std}(r)}$$

### GRPO vs PPO

| 特性 | PPO | GRPO |
|------|-----|------|
| Critic 网络 | 需要（约等于策略模型大小） | 不需要 |
| 优势估计 | GAE（需要 value function） | 组内相对排名 |
| 内存占用 | 高（策略+价值 两个模型） | 低（只需策略模型） |
| KL 约束 | 通过 clip 或 KL penalty | 通过 KL penalty |

In [ ]:
# 环境奖励函数
def simulate_environment_reward(response: str, task_query: str) -> float:
    """
    模拟环境奖励。在真实场景中这会是实际的 API 沙箱。
    奖励信号：格式(+0.3) + 工具相关性(+0.3) + 回答质量(+0.2) + 无幻觉(+0.2)
    """
    reward = 0.0
    if "<tool_call>" in response:
        reward += 0.3  # 使用了工具调用格式
    if len(response) > 50:
        reward += 0.1  # 回答有一定长度
    if len(response) > 150:
        reward += 0.1
    if "\n" in response:
        reward += 0.1  # 有结构
    # 惩罚幻觉
    if any(m in response for m in ["500%", "完全取代", "编造", "虚构"]):
        reward -= 0.3
    # 任务相关性
    if any(kw in response for kw in ["搜索", "计算", "天气", "文件", "结果"]):
        reward += 0.2
    return max(0.0, min(1.0, reward))

# 测试奖励函数
good_response = '<tool_call>{"name": "web_search"}</tool_call>\n根据搜索结果，以下是要点总结...'
bad_response = '我来直接回答，全球GDP增长了500%'
print(f"好回复奖励: {simulate_environment_reward(good_response, 'test'):.2f}")
print(f"坏回复奖励: {simulate_environment_reward(bad_response, 'test'):.2f}")

In [ ]:
# GRPO 训练（简化演示版）
def train_grpo(model, tokenizer, config):
    """Stage 3: GRPO 组相对策略优化"""
    print("\n" + "=" * 60)
    print("Stage 3: GRPO（组相对策略优化）[可选]")
    print("=" * 60)

    device = config.get_device()
    model = model.to(device)

    prompts = [
        "帮我搜索一下大语言模型的最新进展，并总结要点。",
        "请计算 (15 + 27) * 3 - 18 / 6，并解释计算过程。",
        "查看北京的天气，帮我规划明天的出行方案。",
        "搜索中国GDP增长率相关信息，计算关键数据，并将报告保存到 report.txt。",
    ]

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.grpo_lr, weight_decay=0.01)
    ref_params = {name: param.clone().detach() for name, param in model.named_parameters()}

    grpo_rewards = []

    for epoch in range(config.grpo_epochs):
        epoch_reward = 0.0
        for prompt in prompts:
            input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
            responses, rewards = [], []

            # 采样 G 个回复
            model.eval()
            with torch.no_grad():
                for _ in range(config.grpo_group_size):
                    output = model.generate(
                        input_ids, max_new_tokens=200, do_sample=True,
                        temperature=0.8, top_p=0.9,
                        pad_token_id=tokenizer.eos_token_id,
                    )
                    resp = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
                    responses.append(resp)
                    rewards.append(simulate_environment_reward(resp, prompt))
            model.train()

            # 组内相对优势（GRPO 核心）
            rewards_t = torch.tensor(rewards, dtype=torch.float32)
            mean_r, std_r = rewards_t.mean(), rewards_t.std() + 1e-8
            advantages = (rewards_t - mean_r) / std_r

            # 选最优回复做梯度更新
            best_idx = advantages.argmax().item()
            if rewards[best_idx] > 0.3:
                full_text = prompt + responses[best_idx] + tokenizer.eos_token
                enc = tokenizer(full_text, return_tensors="pt", max_length=config.dpo_max_length, truncation=True).to(device)
                outputs = model(**enc, labels=enc["input_ids"])
                loss = outputs.loss * advantages[best_idx].item()
                # KL 惩罚
                kl = sum(((p - ref_params[n]) ** 2).sum() for n, p in model.named_parameters() if n in ref_params)
                total_loss = loss + config.grpo_kl_coeff * kl
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                optimizer.step()
                optimizer.zero_grad()

            epoch_reward += mean_r.item()
            grpo_rewards.append(mean_r.item())
            print(f"  Prompt: {prompt[:30]}... | Mean Reward: {mean_r:.3f} | Best: {max(rewards):.3f}")

        avg_r = epoch_reward / len(prompts)
        print(f"  Epoch {epoch+1} 平均 Reward: {avg_r:.4f}")

    # 保存 GRPO 检查点
    grpo_dir = os.path.join(config.output_dir, "grpo")
    os.makedirs(grpo_dir, exist_ok=True)
    model.save_pretrained(grpo_dir)
    tokenizer.save_pretrained(grpo_dir)
    print(f"  GRPO 模型已保存到 {grpo_dir}")

    return model, grpo_rewards

# 执行 GRPO 训练（可选，如需跳过可注释此行）
model, grpo_rewards = train_grpo(model, tokenizer, config)

In [ ]:
# GRPO 奖励变化
if grpo_rewards:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(grpo_rewards, 'o-', color="#96CEB4", linewidth=2, markersize=6)
    ax.set_xlabel("训练 Prompt")
    ax.set_ylabel("平均 Reward")
    ax.set_title("Stage 3: GRPO 平均奖励变化")
    ax.set_ylim(0, 1.0)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("grpo_rewards.png", dpi=150)
    plt.show()
    print("GRPO 奖励图已保存")

---
## 第六部分：综合评估

在标准化测试任务上对比 **Base → SFT → DPO** 各阶段的模型能力。

评估维度：
- **任务完成率** —— 模型是否成功完成了给定任务
- **工具调用准确率** —— 是否正确选择工具和参数
- **回复质量** —— 回复的完整性、准确性、可读性
- **推理步骤效率** —— 完成任务使用的步骤数是否合理

In [ ]:
import re

# 测试任务集
TEST_TASKS = [
    {"id": "test_001", "query": "帮我搜索一下大语言模型的最新进展，并总结要点。", "expected_tools": ["web_search"], "expected_keywords": ["搜索", "进展", "总结"], "category": "search"},
    {"id": "test_002", "query": "请计算 (15 + 27) * 3 - 18 / 6，并解释计算过程。", "expected_tools": ["calculator"], "expected_keywords": ["计算", "结果"], "category": "math"},
    {"id": "test_003", "query": "读取 report.txt 的内容，然后将摘要写入 summary.txt。", "expected_tools": ["file_read", "file_write"], "expected_keywords": ["读取", "摘要", "写入"], "category": "file"},
    {"id": "test_004", "query": "查看北京的天气，帮我规划明天的出行方案。", "expected_tools": ["get_weather"], "expected_keywords": ["天气", "出行", "建议"], "category": "weather"},
    {"id": "test_005", "query": "搜索中国GDP增长率相关信息，计算关键数据，并将报告保存到 report.txt。", "expected_tools": ["web_search", "calculator", "file_write"], "expected_keywords": ["搜索", "计算", "报告"], "category": "multi_step"},
    {"id": "test_006", "query": "帮我计算 2 的 10 次方加上 3 的 5 次方。", "expected_tools": ["calculator"], "expected_keywords": ["计算", "结果"], "category": "math"},
    {"id": "test_007", "query": "查看上海的天气信息。", "expected_tools": ["get_weather"], "expected_keywords": ["天气", "上海"], "category": "weather"},
    {"id": "test_008", "query": "搜索量子计算的最新突破，写一份简要报告。", "expected_tools": ["web_search"], "expected_keywords": ["量子", "搜索", "报告"], "category": "search"},
    {"id": "test_009", "query": "读取 data.csv 文件的内容并告诉我有什么数据。", "expected_tools": ["file_read"], "expected_keywords": ["读取", "数据"], "category": "file"},
    {"id": "test_010", "query": "帮我查一下深圳今天的天气，然后搜索深圳的旅游景点。", "expected_tools": ["get_weather", "web_search"], "expected_keywords": ["天气", "搜索", "深圳"], "category": "multi_step"},
]

def evaluate_tool_accuracy(response, expected_tools):
    """评估工具调用准确率（F1）"""
    tool_calls = re.findall(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', response, re.DOTALL)
    if not expected_tools:
        return 1.0 if not tool_calls else 0.5
    if not tool_calls:
        return 0.0
    called = set()
    for tc_json in tool_calls:
        try:
            called.add(json.loads(tc_json).get("name", ""))
        except: pass
    if not called:
        return 0.1
    expected_set = set(expected_tools)
    correct = len(expected_set & called)
    prec = correct / len(called) if called else 0
    rec = correct / len(expected_set) if expected_set else 0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def evaluate_quality(response, expected_keywords):
    """评估回复质量"""
    score = 0.0
    if expected_keywords:
        score += 0.4 * sum(1 for kw in expected_keywords if kw in response) / len(expected_keywords)
    length = len(response)
    score += 0.2 * (0 if length < 10 else 0.3 if length < 30 else 1.0 if length < 500 else 0.7 if length < 1000 else 0.5)
    struct = (0.3 if "\n" in response else 0) + (0.4 if any(m in response for m in ["1.", "2.", "- "]) else 0) + (0.3 if "：" in response else 0)
    score += 0.2 * min(1.0, struct)
    score += 0.2 * (0.0 if any(m in response for m in ["500%", "完全取代"]) else 1.0)
    return min(1.0, score)

print(f"定义了 {len(TEST_TASKS)} 个测试任务")

In [ ]:
# 使用模拟输出进行演示评估（避免需要加载多个模型检查点）
stage_outputs = {
    "base": {
        "default": "这个问题很有趣。让我想想...\n我觉得答案可能是这样的。",
    },
    "sft": {
        "search": '<tool_call>{"name": "web_search", "arguments": {"query": "大语言模型最新进展"}}</tool_call>\n\n根据搜索结果，以下是最新进展：\n1. 模型能力持续提升\n2. 多模态能力增强',
        "math": '<tool_call>{"name": "calculator", "arguments": {"expression": "(15+27)*3-18/6"}}</tool_call>\n\n计算结果为123。',
        "file": '<tool_call>{"name": "file_read", "arguments": {"path": "report.txt"}}</tool_call>\n\n<tool_call>{"name": "file_write", "arguments": {"path": "summary.txt", "content": "摘要"}}</tool_call>\n\n已完成文件读取和摘要写入。',
        "weather": '<tool_call>{"name": "get_weather", "arguments": {"city": "北京"}}</tool_call>\n\n天气晴好，建议外出。',
        "multi_step": '<tool_call>{"name": "web_search", "arguments": {"query": "GDP"}}</tool_call>\n\n搜索完成，报告已生成。',
    },
    "dpo": {
        "search": '<tool_call>{"name": "web_search", "arguments": {"query": "大语言模型 最新进展 2024", "max_results": 5}}</tool_call>\n\n根据搜索结果，以下是大语言模型的最新进展要点：\n\n1. **技术突破**：新一代模型在推理能力上取得显著提升\n2. **应用落地**：越来越多企业将大模型应用到实际场景\n3. **开源生态**：开源模型性能逐步接近闭源模型\n\n总结：大语言模型领域正在快速发展。',
        "math": '<tool_call>{"name": "calculator", "arguments": {"expression": "(15+27)*3-18/6"}}</tool_call>\n\n计算结果为 **123.0**。\n\n计算过程：\n1. 先计算括号：15 + 27 = 42\n2. 乘法：42 * 3 = 126\n3. 除法：18 / 6 = 3\n4. 减法：126 - 3 = 123',
        "file": '<tool_call>{"name": "file_read", "arguments": {"path": "report.txt"}}</tool_call>\n\n<tool_call>{"name": "file_write", "arguments": {"path": "summary.txt", "content": "报告摘要：核心要点已提取。"}}</tool_call>\n\n已完成操作：\n1. 读取了 report.txt 的内容\n2. 提取关键信息并生成摘要\n3. 将摘要写入 summary.txt',
        "weather": '<tool_call>{"name": "get_weather", "arguments": {"city": "北京"}}</tool_call>\n\n北京天气：晴，气温12°C，北风3级。\n\n出行建议：\n- 穿着：建议穿外套\n- 交通：天气晴好，适合户外活动\n- 注意：湿度较低，注意补水',
        "multi_step": '<tool_call>{"name": "web_search", "arguments": {"query": "中国GDP增长率 2024"}}</tool_call>\n\n<tool_call>{"name": "calculator", "arguments": {"expression": "100 * 1.05 ** 5"}}</tool_call>\n\n<tool_call>{"name": "file_write", "arguments": {"path": "report.txt", "content": "GDP增长率研究报告"}}</tool_call>\n\n已完成研究报告：\n1. 搜索了最新GDP相关信息\n2. 计算了关键增长数据\n3. 将完整报告保存到 report.txt',
    },
}

# 评估各阶段
all_results = {}
for stage_name, outputs in stage_outputs.items():
    print(f"\n{'='*60}")
    print(f"评估阶段: {stage_name}")
    print(f"{'='*60}")
    results = []
    for task in TEST_TASKS:
        response = outputs.get(task["category"], outputs.get("default", ""))
        tool_acc = evaluate_tool_accuracy(response, task["expected_tools"])
        quality = evaluate_quality(response, task["expected_keywords"])
        completed = len(response) > 20 and any(kw in response for kw in task["expected_keywords"])
        results.append({"task_id": task["id"], "category": task["category"], "completed": completed, "tool_acc": tool_acc, "quality": quality})
        print(f"  {task['id']}: 完成={'是' if completed else '否'} | 工具={tool_acc:.2f} | 质量={quality:.2f}")
    all_results[stage_name] = results

# 汇总
print(f"\n{'='*60}")
print("评估汇总")
print(f"{'='*60}")
print(f"{'阶段':<10} {'完成率':<12} {'工具准确率':<12} {'回复质量':<12}")
print("-" * 46)
for stage, results in all_results.items():
    n = len(results)
    comp = sum(r["completed"] for r in results) / n
    tool = sum(r["tool_acc"] for r in results) / n
    qual = sum(r["quality"] for r in results) / n
    print(f"{stage:<10} {comp:<12.2%} {tool:<12.2f} {qual:<12.2f}")

In [ ]:
# 可视化：各阶段对比柱状图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stages = list(all_results.keys())
metrics_names = ["完成率", "工具准确率", "回复质量"]
colors = ["#4ECDC4", "#FF6B6B", "#45B7D1"]

# 图1：各阶段整体指标柱状图
ax1 = axes[0]
x = np.arange(len(metrics_names))
width = 0.8 / len(stages)
for i, stage in enumerate(stages):
    results = all_results[stage]
    n = len(results)
    values = [
        sum(r["completed"] for r in results) / n,
        sum(r["tool_acc"] for r in results) / n,
        sum(r["quality"] for r in results) / n,
    ]
    ax1.bar(x + i * width, values, width, label=stage, color=colors[i % len(colors)], alpha=0.85)
ax1.set_xticks(x + width)
ax1.set_xticklabels(metrics_names)
ax1.set_ylabel("分数")
ax1.set_title("各阶段整体指标对比")
ax1.set_ylim(0, 1.1)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# 图2：性能趋势
ax2 = axes[1]
for stage in stages:
    results = all_results[stage]
    n = len(results)
comp_by_stage = [sum(r["completed"] for r in all_results[s]) / len(all_results[s]) for s in stages]
tool_by_stage = [sum(r["tool_acc"] for r in all_results[s]) / len(all_results[s]) for s in stages]
qual_by_stage = [sum(r["quality"] for r in all_results[s]) / len(all_results[s]) for s in stages]

ax2.plot(stages, comp_by_stage, 'o-', color="#FF6B6B", linewidth=2, markersize=8, label="完成率")
ax2.plot(stages, tool_by_stage, 's-', color="#4ECDC4", linewidth=2, markersize=8, label="工具准确率")
ax2.plot(stages, qual_by_stage, '^-', color="#45B7D1", linewidth=2, markersize=8, label="回复质量")
ax2.set_xlabel("训练阶段")
ax2.set_ylabel("分数")
ax2.set_title("训练阶段性能趋势")
ax2.set_ylim(0, 1.1)
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle("Agent 模型各阶段综合评估", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("evaluation_comparison.png", dpi=150)
plt.show()
print("评估对比图已保存")

---
## 第七部分：课程总结

### 完整训练路径图

```
┌──────────────────────────────────────────────────────────────────────┐
│                     LLM Agent 完整训练路径                          │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ┌─────────┐     ┌─────────┐     ┌─────────┐     ┌─────────┐       │
│  │ 基座模型 │ ──→ │  SFT    │ ──→ │  DPO    │ ──→ │ GRPO/RL │       │
│  │ (预训练) │     │(有监督) │     │(偏好)   │     │(强化)   │       │
│  └─────────┘     └─────────┘     └─────────┘     └─────────┘       │
│       │               │               │               │              │
│  大规模语料      对话+工具数据     好/坏轨迹对      环境奖励信号     │
│  Next Token      Loss Masking     无需RM          组相对优势       │
│  预测             LoRA 微调       β 约束          KL 惩罚          │
│                                                                      │
│  能力: 语言理解  →  格式+工具  →  质量+偏好  →  环境适应          │
└──────────────────────────────────────────────────────────────────────┘
```

### 5 个核心验证问题

#### 问题 1：训练一个模型的完整步骤是什么？

**完整答案：**

1. **数据准备** —— 收集/生成高质量训练数据，确定格式（ChatML、tool_calls 等）
2. **选择基座模型** —— 根据资源和场景选择合适的预训练模型（如 Qwen2-0.5B）
3. **SFT（有监督微调）** —— 用标注数据教会模型对话格式和基本能力
4. **偏好对齐（DPO/RLHF）** —— 用偏好数据让模型学会区分好坏回复
5. **强化学习（可选）** —— 在环境中用 GRPO/PPO 继续优化
6. **评估与迭代** —— 用标准化测试集评估，根据结果调整数据和策略

#### 问题 2：SFT 的数据需要什么格式？为什么要做 Loss Masking？

**完整答案：**

- **数据格式**：多轮对话格式（system/user/assistant/tool），工具调用使用 `tool_calls` 字段
- **Loss Masking 的原因**：
  - 我们只想让模型学会 **"如何回复"**（assistant 部分）
  - system/user/tool 部分设置 label=-100，被 CrossEntropy 忽略
  - 如果不做 masking，模型会把 user 的提问方式也学进去，导致行为异常
  - 类似于「老师只在答题部分打分，不在题目部分打分」

#### 问题 3：DPO 相比 RLHF 的核心简化是什么？

**完整答案：**

- **RLHF 流程**：标注偏好 → 训练奖励模型(RM) → 用 PPO 优化策略 → KL 约束
- **DPO 流程**：标注偏好 → 直接优化策略（跳过 RM 和 PPO）
- **核心简化**：DPO 证明了在 Bradley-Terry 模型假设下，最优策略可以用 **闭式解** 表示，不需要显式的奖励模型。损失函数直接用偏好对的 log probability ratio 构造。
- **实际好处**：训练更简单稳定、不需要采样、超参数更少、不需要额外的 RM 模型

#### 问题 4：GRPO 相比 PPO 省掉了什么？AgentRL 的奖励设计难在哪？

**完整答案：**

- **GRPO 省掉了 Critic 网络**：
  - PPO 需要一个 value network（通常与策略模型同等大小）来估计状态价值
  - GRPO 通过「组内相对排名」替代：对每个 prompt 采样 G 个回复，用 (r_i - mean) / std 计算相对优势
  - 节省了约 50% 的内存开销

- **AgentRL 奖励设计的难点**：
  - **稀疏性**：只有任务完成后才有明确奖励，中间步骤难以评估
  - **信用分配**：一个包含 5 步工具调用的轨迹，错在哪一步？
  - **环境不确定性**：工具返回结果可能变化，相同操作不一定得到相同奖励
  - **多维度平衡**：正确性、效率、格式规范之间需要权衡

#### 问题 5：从基座模型到 Agent 模型，经历了哪几个训练阶段？

**完整答案：**

| 阶段 | 输入 | 输出 | 核心能力变化 |
|------|------|------|-------------|
| 预训练 | 大规模语料 | 基座模型 | 语言理解与生成 |
| SFT | 对话+工具数据 | SFT 模型 | +对话格式 +Function Calling |
| DPO | 偏好对数据 | DPO 模型 | +偏好质量 +正确工具选择 |
| GRPO/RL | 环境交互 | Agent 模型 | +环境适应 +任务完成率 |

每个阶段都是在前一阶段的基础上**增量学习**新能力，而不是从零开始。这就是为什么训练顺序很重要：先学格式（SFT），再学偏好（DPO），最后在环境中优化（RL）。

### 下一步学习建议

完成 14 天的学习后，你已经掌握了 LLM 训练的核心技能。以下是进阶方向：

| 方向 | 内容 | 推荐资源 |
|------|------|----------|
| **多模态** | 训练能理解图片/音频的模型 | LLaVA、Qwen-VL |
| **长上下文** | 扩展模型的上下文窗口 | RoPE 外推、YaRN |
| **推理能力** | 专注数学/代码推理训练 | DeepSeek-R1、o1 复现 |
| **高效训练** | 分布式训练、混合精度 | DeepSpeed、FSDP |
| **部署优化** | 模型量化、推理加速 | GPTQ、AWQ、vLLM |
| **安全对齐** | 红队测试、Constitutional AI | Anthropic 的对齐研究 |

### 给 Agent 开发者的寄语

**「从基座到 Agent，是模型从『知道』到『会做』的跨越。」**

在这 14 天的学习中，我们走过了一条完整的路径：

- **Day 1-2**：理解数据是如何被模型「看到」的（Tokenizer）
- **Day 3-4**：理解模型是如何「思考」的（Transformer）
- **Day 5-6**：教会模型「说话」（SFT）
- **Day 7-8**：教会模型「说好话」（DPO）
- **Day 9-10**：教会模型「自我进化」（RL）
- **Day 11-12**：教会模型「使用工具」（Agent RL）
- **Day 13-14**：将一切串联成完整 Pipeline

**核心心得：**

1. **数据质量 > 数据数量** —— 50 条高质量轨迹 > 5000 条低质量数据
2. **训练顺序很重要** —— 先学格式，再学偏好，最后环境优化
3. **评估驱动迭代** —— 没有评估就没有改进方向
4. **从小模型开始** —— 0.5B 模型能验证所有关键概念
5. **理解原理再用框架** —— 手写一遍 DPO loss 比调 10 次 trl 参数更有价值

**祝你在 LLM 训练的道路上越走越远！**

In [ ]:
# 课程完成标记
print("=" * 60)
print("  14 天 LLM 训练冲刺营 —— 课程完成！")
print("=" * 60)
print()
print("  完成内容：")
print("  [x] Stage 1: SFT 有监督微调")
print("  [x] Stage 2: DPO 直接偏好优化")
print("  [x] Stage 3: GRPO 强化学习（可选）")
print("  [x] 综合评估与可视化")
print("  [x] 5 个核心验证问题")
print()
print("  从基座模型到 Agent，你已经走完了完整的训练路径。")
print("  Keep learning, keep building!")